# الفصل 15: خليط الخبراء: Mixture of Experts

> تخيّل أنك تستطيع بناء نموذج بتريليون معامل، لكن عند معالجة أي رمز واحد، لا يُستخدَم فعليًا سوى جزء صغير منه (مثلًا 37 مليار معامل فقط) — هذا بالضبط ما تحققه معمارية Mixture of Experts (MoE).

- **الجزء:** 4 — المعماريات الحديثة
- **الفصل:** 15
- **المستوى:** متقدم
- **زمن القراءة:** 20 دقيقة
- **المتطلبات:** الفصل 14: الترميز الموضعي المتقدم وآليات الانتباه الفعّالة
- **الوسوم:** MoE، Routing، Load Balancing، Mixtral

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch15-mixture-of-experts.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-04-modern-architectures%2Fch15-mixture-of-experts.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-04-modern-architectures/ch15-mixture-of-experts.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch15-mixture-of-experts.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch15-mixture-of-experts.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch15-mixture-of-experts.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch15-mixture-of-experts.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-04-modern-architectures%2Fch15-mixture-of-experts.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-04-modern-architectures/ch15-mixture-of-experts.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch15-mixture-of-experts.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch15-mixture-of-experts.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch15-mixture-of-experts.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> تخيّل أنك تستطيع بناء نموذج بتريليون معامل، لكن عند معالجة أي رمز واحد، لا يُستخدَم فعليًا سوى جزء صغير منه (مثلًا 37 مليار معامل فقط) — هذا بالضبط ما تحققه معمارية Mixture of Experts (MoE).

> **بيانات الفصل**
>
> - **الجزء:** 4 — المعماريات الحديثة
> - **المستوى:** متقدم
> - **زمن القراءة:** 20 دقيقة
> - **المتطلبات:** الفصل 14: الترميز الموضعي المتقدم وآليات الانتباه الفعّالة


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم MoE
- فهم Routing
- فهم Load Balancing


## 1. لماذا يهمّك هذا الموضوع؟

تخيّل أنك تستطيع بناء نموذج بتريليون معامل، لكن عند معالجة أي رمز واحد، لا يُستخدَم فعليًا سوى جزء صغير منه (مثلًا 37 مليار معامل فقط) — هذا بالضبط ما تحققه معمارية **Mixture of Experts (MoE)**. هذا الفصل يشرح كيف نجحت نماذج مثل **Mixtral 8x7B** و**DeepSeek-V3** في فصل **سعة النموذج** (capacity، أي إجمالي عدد المعاملات) عن **التكلفة الحسابية الفعلية** (compute، أي عدد المعاملات المُفعَّلة فعليًا لكل رمز)، محققة أداءً يضاهي نماذج كثيفة (dense) أضخم بكثير، بتكلفة استدلال أقرب لنموذج أصغر بكثير.

---


## 2. الخريطة الذهنية


```text
الفكرة الأساسية: بدلاً من طبقة FeedForward واحدة كبيرة،
استخدم عدة "خبراء" (experts) صغيرة، وفعّل فقط عددًا قليلًا منها لكل رمز

البنية:
│
├── Router (بوابة التوجيه): شبكة صغيرة تقرر، لكل رمز، أي الخبراء
│   الأنسب لمعالجته (عادة أفضل K من أصل N خبيرًا، K=1 أو 2)
│
├── N خبير (Experts): كل منها شبكة FeedForward مستقلة كاملة
│   (بنفس بنية FFN العادية، لكن بمعاملات منفصلة لكل خبير)
│
└── الدمج: مخرجات الخبراء المُختارين تُدمَج موزونة بأوزان الـ Router

المشكلة الحرجة: Load Balancing (توازن الحمل)
│
├── بدون تدخل، يميل الـ Router لتفضيل عدد قليل من الخبراء
│   ("الخبراء الأقوياء يصبحون أقوى، البقية مهمَلة" - rich get richer)
│
├── الحل: Load Balancing Loss (دالة خسارة إضافية)
│   تُعاقب النموذج إذا كان التوزيع على الخبراء غير متوازن
│
└── بدون هذا التوازن، تنهار فائدة MoE بالكامل (بعض الخبراء لا يتعلم شيئًا)

أمثلة حقيقية:
│
├── Mixtral 8x7B: 8 خبراء، Top-2 تفعيل
│   47B معامل إجمالي، لكن ~13B فقط مُفعَّل لكل رمز
│
└── DeepSeek-V3: معمارية MoE متقدمة بخبراء "مشتركين" (shared experts)
    دائمي التفعيل + خبراء متخصصين (routed) يُختارون ديناميكيًا
    671B معامل إجمالي، لكن ~37B فقط مُفعَّل لكل رمز
```


---


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

تخيّل مستشفى كبيرًا فيه 50 طبيبًا اختصاصيًا. عندما يأتي مريض، لا يفحصه كل الخمسين طبيبًا معًا (هذا مُكلف ومهدر للوقت) — بل تُوجّهه ممرضة الاستقبال (الـ Router) إلى أنسب طبيبين أو ثلاثة لحالته المحددة (مثلًا: طبيب قلب وطبيب غدد صماء). المستشفى ككل يملك "سعة" هائلة من الخبرة (50 طبيبًا)، لكن "تكلفة" كل حالة محددة فعليًا محدودة بعدد الأطباء المُستشارين الفعليين لها. هذا بالضبط ما تحققه MoE: سعة إجمالية هائلة (كل الخبراء مجتمعين)، لكن تكلفة حسابية محدودة (فقط الخبراء المُفعَّلون لكل رمز محدد).

### 🟡 طبقة الآلية: الصيغة الرياضية الكاملة

**طبقة Router (بوابة التوجيه)**:

لكل رمز بتمثيل $x \in \mathbb{R}^{d_{model}}$، تحسب بوابة التوجيه درجات توافق مع كل خبير عبر طبقة خطية بسيطة:

$$g(x) = \text{softmax}(W_g x)$$

حيث $W_g \in \mathbb{R}^{N \times d_{model}}$ و$N$ عدد الخبراء. نختار أعلى K قيمة (عادة K=1 أو K=2) من $g(x)$:

$$\mathcal{T} = \text{TopK}(g(x), K)$$

**مخرج طبقة MoE**:

$$y = \sum_{i \in \mathcal{T}} g_i(x) \cdot E_i(x)$$

حيث $E_i(x)$ هو مخرج الخبير رقم $i$ (شبكة FeedForward كاملة مستقلة)، و$g_i(x)$ هو وزن الـ Router المُطبَّع (إعادة تطبيع softmax على الخبراء المُختارين فقط).

**Load Balancing Loss**: لتفادي "انهيار الخبراء" (expert collapse) حيث يُفضَّل عدد قليل من الخبراء باستمرار، نُضيف دالة خسارة مساعدة تُشجِّع التوزيع المتساوي:

$$\mathcal{L}_{balance} = N \cdot \sum_{i=1}^{N} f_i \cdot P_i$$

حيث:
- $f_i$ = نسبة الرموز الفعلية المُوجَّهة للخبير $i$ عبر الدفعة (fraction of tokens dispatched)
- $P_i$ = متوسط احتمال الـ Router لاختيار الخبير $i$ عبر الدفعة

هذا الضرب بين $f_i$ و$P_i$ (بدلًا من استخدام أحدهما فقط) حيلة رياضية ذكية: إذا كان خبير معين يحصل على نسبة عالية من الرموز **و**احتمال توجيه عالٍ باستمرار، تزداد الخسارة بشكل كبير (تناسب تربيعي تقريبًا)، مما يدفع النموذج لتوزيع الحمل بشكل أكثر تساويًا.

$$\mathcal{L}_{total} = \mathcal{L}_{task} + \alpha \cdot \mathcal{L}_{balance}$$

حيث $\alpha$ معامل موازنة صغير (عادة حوالي 0.01).

### 🔴 طبقة التعمّق: الكود الكامل


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# 1. خبير واحد (Expert) — شبكة FeedForward قياسية
# ============================================================
class Expert(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)

# ============================================================
# 2. طبقة MoE كاملة مع Top-K Routing وLoad Balancing
# ============================================================
class MoELayer(nn.Module):
    def __init__(self, d_model: int, d_ff: int, num_experts: int = 8,
                 top_k: int = 2, dropout: float = 0.1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.d_model = d_model

        self.experts = nn.ModuleList([
            Expert(d_model, d_ff, dropout) for _ in range(num_experts)
        ])
        self.router = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        العائد: (output, aux_loss)
        """
        batch_size, seq_len, d_model = x.shape
        x_flat = x.view(-1, d_model)   # (batch*seq_len, d_model) لتبسيط التوجيه

        # 1. حساب درجات الـ Router
        router_logits = self.router(x_flat)              # (N_tokens, num_experts)
        router_probs = F.softmax(router_logits, dim=-1)

        # 2. اختيار أفضل K خبراء لكل رمز
        top_k_probs, top_k_indices = torch.topk(router_probs, self.top_k, dim=-1)
        # إعادة تطبيع الاحتمالات على الخبراء المُختارين فقط
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

        # 3. حساب مخرج كل رمز بتجميع مخرجات الخبراء المُختارين (موزونة)
        output = torch.zeros_like(x_flat)

        for expert_idx in range(self.num_experts):
            # أي الرموز اختارت هذا الخبير ضمن أفضل K؟
            expert_mask = (top_k_indices == expert_idx).any(dim=-1)  # (N_tokens,)

            if expert_mask.sum() == 0:
                continue  # لا رموز لهذا الخبير في هذه الدفعة

            # الوزن المقابل لهذا الخبير لكل رمز اختاره
            expert_weight_positions = (top_k_indices == expert_idx)
            expert_weights = (top_k_probs * expert_weight_positions).sum(dim=-1)  # (N_tokens,)

            expert_input = x_flat[expert_mask]
            expert_output = self.experts[expert_idx](expert_input)

            output[expert_mask] += expert_weights[expert_mask].unsqueeze(-1) * expert_output

        output = output.view(batch_size, seq_len, d_model)

        # 4. حساب Load Balancing Loss
        aux_loss = self._compute_load_balancing_loss(router_probs, top_k_indices)

        return output, aux_loss

    def _compute_load_balancing_loss(self, router_probs, top_k_indices):
        """
        L_balance = N * Σ f_i * P_i
        f_i: نسبة الرموز الفعلية المُوجَّهة للخبير i
        P_i: متوسط احتمال الـ Router للخبير i عبر كل الرموز
        """
        num_tokens = router_probs.shape[0]

        # P_i: متوسط احتمال الـ router لكل خبير (عبر كل الرموز، وليس فقط المُختارة)
        P = router_probs.mean(dim=0)   # (num_experts,)

        # f_i: نسبة الرموز التي اختارت هذا الخبير ضمن Top-K فعليًا
        expert_mask = F.one_hot(top_k_indices, num_classes=self.num_experts).float()
        # expert_mask: (num_tokens, top_k, num_experts) → نجمع على top_k
        f = expert_mask.sum(dim=1).mean(dim=0) / self.top_k  # (num_experts,)

        aux_loss = self.num_experts * torch.sum(f * P)
        return aux_loss

# ============================================================
# 3. طبقة Transformer كاملة تستخدم MoE بدلاً من FFN القياسية
# ============================================================
class TransformerBlockWithMoE(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_experts=8, top_k=2, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)  # من الفصل 10
        self.ln2 = nn.LayerNorm(d_model)
        self.moe = MoELayer(d_model, d_ff, num_experts, top_k, dropout)

    def forward(self, x, mask=None):
        normed = self.ln1(x)
        attn_out, _ = self.attn(normed, normed, normed, mask)
        x = x + attn_out

        normed = self.ln2(x)
        moe_out, aux_loss = self.moe(normed)
        x = x + moe_out

        return x, aux_loss

# ============================================================
# 4. مفهوم DeepSeek-V3 المتقدم: خبراء مشتركون + خبراء متخصصون
# ============================================================
class DeepSeekStyleMoELayer(nn.Module):
    """
    تطبيق مبسّط لفكرة DeepSeek-V3: خبراء مشتركون (shared experts)
    مُفعَّلون دائمًا لكل رمز (يلتقطون المعرفة العامة المشتركة)،
    بالإضافة لخبراء متخصصين (routed experts) يُختارون ديناميكيًا.
    """
    def __init__(self, d_model, d_ff, num_shared_experts=2,
                 num_routed_experts=64, top_k_routed=6, dropout=0.1):
        super().__init__()

        # خبراء مشتركون: نشطون دائمًا لكل رمز، بدون توجيه
        self.shared_experts = nn.ModuleList([
            Expert(d_model, d_ff, dropout) for _ in range(num_shared_experts)
        ])

        # خبراء متخصصون: يُختار منهم عبر Top-K Routing
        self.routed_moe = MoELayer(
            d_model, d_ff, num_experts=num_routed_experts,
            top_k=top_k_routed, dropout=dropout
        )

    def forward(self, x):
        # مساهمة الخبراء المشتركين (دائمًا نشطة)
        shared_output = sum(expert(x) for expert in self.shared_experts)

        # مساهمة الخبراء المتخصصين (عبر التوجيه الديناميكي)
        routed_output, aux_loss = self.routed_moe(x)

        return shared_output + routed_output, aux_loss

def compute_moe_efficiency_stats(num_experts, top_k, expert_params, router_params):
    """حساب نسبة السعة الكلية إلى التكلفة الفعلية في MoE."""
    total_params = num_experts * expert_params + router_params
    active_params = top_k * expert_params + router_params

    efficiency_ratio = total_params / active_params

    print(f"إجمالي المعاملات (السعة): {total_params:,}")
    print(f"المعاملات النشطة لكل رمز (التكلفة): {active_params:,}")
    print(f"نسبة السعة إلى التكلفة: {efficiency_ratio:.2f}x")

    return total_params, active_params, efficiency_ratio


---


## 4. مثال محلول خطوة بخطوة

لنحاكِ حساب Load Balancing Loss لدفعة من 4 رموز مع 4 خبراء، top_k=1 (أبسط حالة توضيحية).

**احتمالات الـ Router لكل رمز** (بعد softmax، 4 رموز × 4 خبراء):

$$P = \begin{bmatrix} 0.7 & 0.1 & 0.1 & 0.1 \\ 0.6 & 0.2 & 0.1 & 0.1 \\ 0.8 & 0.1 & 0.05 & 0.05 \\ 0.65 & 0.15 & 0.1 & 0.1 \end{bmatrix}$$

(لاحظ: كل الرموز الأربعة "تُفضِّل" بشدة الخبير الأول — سيناريو عدم توازن واضح)

**متوسط احتمال الـ Router لكل خبير ($P_i$)**:

$$P_1 = \frac{0.7+0.6+0.8+0.65}{4} = 0.6875, \quad P_2 = \frac{0.1+0.2+0.1+0.15}{4} = 0.1375$$
$$P_3 = \frac{0.1+0.1+0.05+0.1}{4} = 0.0875, \quad P_4 = \frac{0.1+0.1+0.05+0.1}{4} = 0.0875$$

**نسبة الرموز الفعلية لكل خبير ($f_i$)** (بما أن top_k=1، كل رمز يختار فقط الأعلى احتمالًا، وهو الخبير 1 في كل الحالات الأربع):

$$f_1 = \frac{4}{4} = 1.0, \quad f_2 = f_3 = f_4 = 0.0$$

**حساب Load Balancing Loss**:

$$\mathcal{L}_{balance} = N \sum_i f_i P_i = 4 \times (1.0 \times 0.6875 + 0 + 0 + 0) = 4 \times 0.6875 = 2.75$$

**مقارنة بسيناريو متوازن مثالي** (لو كان كل خبير يحصل على ربع الرموز وربع الاحتمال، أي $f_i = P_i = 0.25$ لكل خبير):

$$\mathcal{L}_{balance}^{ideal} = 4 \times (4 \times 0.25 \times 0.25) = 4 \times 0.25 = 1.0$$

**التفسير**: الخسارة في السيناريو غير المتوازن (2.75) أكبر بكثير من السيناريو المثالي (1.0، وهو أصغر قيمة ممكنة لهذه الصيغة عمومًا عند N خبراء متساوين). هذا يُظهر عدديًا كيف "تُعاقب" دالة الخسارة هذه حالات تركز الحمل على خبير واحد، مما يدفع عملية التحسين (optimization) لتعلّم توزيع أكثر توازنًا عبر الخبراء خلال التدريب.

---


## 5. نصائح وحيل

> 💡 **ابدأ بـ top_k=2**: هذا الاختيار (مستخدَم في Mixtral وGShard) يوازن بين الكفاءة الحسابية والجودة — top_k=1 أكثر كفاءة لكن أكثر هشاشة في التدريب، وtop_k أكبر يُقلِّل فائدة MoE الأساسية (توفير الحساب).

> 💡 **معامل $\alpha$ لـ Load Balancing Loss يحتاج ضبطًا دقيقًا**: قيمة كبيرة جدًا تُجبر توزيعًا متساويًا مصطنعًا يُضعف التخصص المفيد للخبراء؛ قيمة صغيرة جدًا تسمح بانهيار الخبراء. القيمة الشائعة في الأدبيات حوالي 0.01.

> 💡 **خبراء مشتركون (shared experts) فكرة قوية من DeepSeek**: الاحتفاظ بخبير أو خبيرين "عامّين" نشطين دائمًا (بالإضافة للخبراء المُوجَّهين ديناميكيًا) يُحسِّن الاستقرار لأنه يضمن وجود مسار معرفي أساسي مشترك لكل الرموز، بينما يترك التخصص الدقيق للخبراء الموجَّهين.

> 💡 **راقب توزيع استخدام الخبراء أثناء التدريب**: ارسم histogram لعدد الرموز المُوجَّهة لكل خبير بشكل دوري — إذا رأيت خبيرًا أو أكثر لا يكاد يُستخدَم إطلاقًا، فهذه إشارة واضحة لمشكلة في معامل الموازنة أو تهيئة الـ Router.

---


## 6. أخطاء شائعة ومصائد

> ⚠️ **إهمال Load Balancing Loss كليًا**: يؤدي هذا بشكل شبه مؤكد إلى "انهيار الخبراء" حيث يُصبح معظم الحمل على 1-2 خبير فقط، بينما البقية لا تتعلم شيئًا مفيدًا — هذا يُلغي عمليًا كل فائدة استخدام MoE من الأساس.

> ⚠️ **التعامل مع MoE كأنه "مجاني" حسابيًا بالمطلق**: رغم أن التكلفة الحسابية الفعلية (FLOPs) لكل رمز محدودة، فإن **البصمة الذاكرية الإجمالية** للنموذج (كل الخبراء يجب تحميلهم في الذاكرة، حتى غير المُفعَّلين لرمز معين) تبقى كبيرة جدًا، تساوي تقريبًا نموذجًا كثيفًا بنفس إجمالي المعاملات — هذا قيد مهم غالبًا ما يُغفَل عند التخطيط لمتطلبات النشر.

> ⚠️ **عدم التعامل بعناية مع التوزيع غير المتساوي للرموز بين الأجهزة (في التدريب الموزَّع)**: إذا تم توزيع الخبراء المختلفين على أجهزة GPU مختلفة (expert parallelism)، فإن عدم توازن التوجيه يعني أن بعض الأجهزة ستكون محمَّلة بشدة بينما أخرى خاملة تقريبًا — هذا يتطلب تقنيات إضافية (مثل capacity factor وtoken dropping) لم نتطرق لتفاصيلها هنا لكنها ضرورية في الإنتاج الفعلي على نطاق واسع.

---


## 7. جدول مقارنة

| الخاصية | النموذج الكثيف (Dense) | نموذج MoE |
|---|---|---|
| **تفعيل المعاملات** | كل المعاملات نشطة لكل رمز | فقط جزء صغير (Top-K من N خبير) |
| **السعة الإجمالية مقابل التكلفة** | متساويتان دائمًا | السعة أكبر بكثير من التكلفة الفعلية |
| **تعقيد التدريب** | أبسط نسبيًا | أعقد (يحتاج Load Balancing، توزيع خبراء) |
| **البصمة الذاكرية** | تساوي حجم النموذج تمامًا | كبيرة (كل الخبراء محمَّلون) رغم تفعيل جزئي |
| **أمثلة** | GPT-3, LLaMA-2 (النسخ الأساسية) | Mixtral 8x7B, DeepSeek-V3, Switch Transformer |
| **الأداء مقابل الحساب الفعلي** | متناسب تقريبًا | أفضل بكثير (أداء نموذج كبير بحساب نموذج أصغر) |

---


## 8. ملخص أفضل الممارسات

- ✅ لا تهمل أبدًا Load Balancing Loss عند تدريب أي معمارية MoE من الصفر.
- ✅ ابدأ بـ top_k=2 وعدد خبراء معتدل (8) كنقطة انطلاق قياسية مثبتة الفعالية.
- ✅ فكّر في إضافة خبراء مشتركين (shared experts) دائمي التفعيل لتحسين استقرار التدريب.
- ✅ خطّط لمتطلبات الذاكرة بناءً على **إجمالي** معاملات كل الخبراء، وليس فقط المُفعَّل منها.
- ✅ راقب توزيع استخدام الخبراء بانتظام كأداة تشخيصية لصحة التدريب.

---


## 9. أسئلة مقابلات

**س1: اشرح الفكرة الأساسية وراء Mixture of Experts، وكيف تسمح بفصل سعة النموذج عن تكلفته الحسابية.**

**الإجابة النموذجية**: في الشبكة العصبية الكثيفة التقليدية، كل معامل في النموذج يُستخدَم في معالجة كل رمز مُدخَل — أي أن سعة النموذج (إجمالي عدد المعاملات) وتكلفته الحسابية الفعلية (عدد العمليات المطلوبة لكل رمز) متلازمتان تمامًا: لا يمكن زيادة إحداهما دون الأخرى. تكسر Mixture of Experts هذا الترابط عبر استبدال طبقة FeedForward الواحدة الكبيرة بعدة طبقات FeedForward مستقلة أصغر ("خبراء")، مع شبكة توجيه صغيرة (Router) تُقرر، لكل رمز على حدة، أي عدد قليل من الخبراء (عادة 1 أو 2 من أصل العشرات) يجب تفعيلهم فعليًا لمعالجة هذا الرمز بالذات. هذا يعني أن النموذج ككل قد يحتوي مئات المليارات من المعاملات (سعة هائلة، موزعة عبر كل الخبراء)، لكن عدد المعاملات **النشطة فعليًا** لمعالجة أي رمز واحد محدود جدًا (فقط الخبراء المُختارين + الـ Router)، مما يسمح بتحقيق أداء مماثل لنموذج كثيف ضخم جدًا، بتكلفة حسابية فعلية أقرب بكثير لنموذج أصغر حجمًا.

**س2: لماذا تُعتبر Load Balancing Loss ضرورية في تدريب MoE، وماذا يحدث إذا أُهمِلت؟**

**الإجابة النموذجية**: بدون أي تدخل إضافي، تميل عملية التحسين (optimization) في بداية التدريب لتفضيل عدد قليل من الخبراء بشكل عشوائي بسيط (ربما بسبب تهيئة عشوائية أعطتهم ميزة طفيفة في البداية) — وبما أن هؤلاء الخبراء يحصلون على المزيد من الرموز للتدريب عليها، فإنهم يتحسنون أسرع، مما يجعل الـ Router يُفضِّلهم أكثر فأكثر (ظاهرة "الغني يزداد غنى" أو rich-get-richer)، بينما يبقى باقي الخبراء شبه معطَّلين ولا يتعلمون أي شيء مفيد تقريبًا، لأنهم لا يحصلون على بيانات كافية للتدريب. هذا يُسمى "انهيار الخبراء" (expert collapse)، وهو يُلغي عمليًا أي فائدة من استخدام معمارية MoE، حيث يتصرف النموذج فعليًا كأنه نموذج كثيف أصغر بكثير (بحجم عدد الخبراء النشطين فقط) رغم امتلاكه سعة إجمالية أكبر بكثير. دالة Load Balancing Loss تُضيف حافزًا تدريبيًا صريحًا يُعاقب أي تركّز غير متساوٍ للحمل على الخبراء (عبر ضرب نسبة الرموز الفعلية وفي متوسط احتمال الـ Router لكل خبير، ثم جمع هذا عبر كل الخبراء)، مما يدفع النموذج لتعلّم توزيع أكثر توازنًا يستفيد من كل الخبراء المتاحين فعليًا.

**س3: ما الفرق بين الخبراء "المشتركين" (shared experts) والخبراء "الموجَّهين" (routed experts) في معماريات مثل DeepSeek-V3؟**

**الإجابة النموذجية**: الخبراء الموجَّهون (routed experts) هم الخبراء التقليديون في MoE: يُختار عدد قليل منهم ديناميكيًا لكل رمز عبر شبكة التوجيه (Router)، بناءً على محتوى الرمز نفسه — كل خبير موجَّه يتخصص (نظريًا) في نمط معين من البيانات أو المهام. أما الخبراء المشتركون (shared experts)، فهم عدد صغير وثابت من الخبراء **يُفعَّلون دائمًا لكل رمز دون أي توجيه أو اختيار**، تمامًا كطبقة FeedForward تقليدية في نموذج كثيف. الحدس وراء هذا التصميم الهجين: هناك أنماط ومعرفة لغوية عامة ومشتركة يحتاجها كل رمز تقريبًا (مثل القواعد النحوية الأساسية، فهم بنية الجملة)، ومن غير الكفؤ أن يُعاد تعلّم هذه المعرفة العامة بشكل مكرر ومستقل داخل كل خبير موجَّه على حدة. بفصل هذه المعرفة العامة إلى خبراء مشتركين دائمي التفعيل، يصبح بإمكان الخبراء الموجَّهين التركيز حصريًا على التخصص الدقيق والتمايز فيما بينهم، مما يُحسِّن كفاءة استخدام سعة النموذج الإجمالية ويُحسِّن استقرار التدريب بشكل ملحوظ، كما أظهرت أبحاث DeepSeek.

---


## 10. تمرين عملي

1. نفّذ `MoELayer` كاملة بـ4 خبراء وtop_k=2، وطبّقها على دفعة عشوائية من الرموز، وتحقق من شكل المخرج النهائي ومن أن `aux_loss` رقم موجب معقول.
2. صمّم تجربة تدريب مصغّرة (بيانات اصطناعية بسيطة) لمقارنة سلوك الخسارة وتوزيع استخدام الخبراء بين حالتين: مع Load Balancing Loss (α=0.01) وبدونها (α=0) — ارسم histogram لاستخدام الخبراء في الحالتين.
3. احسب `compute_moe_efficiency_stats` لمعمارية شبيهة بـ Mixtral (8 خبراء، top_k=2) وقارنها مع معمارية شبيهة بـ DeepSeek-V3 (مئات الخبراء، top_k أكبر نسبيًا، مع خبراء مشتركين) من حيث نسبة السعة إلى التكلفة.

---


## 11. مراجع للتعمّق

- Shazeer, N., et al. (2017). *Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer*.
- Fedus, W., et al. (2021). *Switch Transformers: Scaling to Trillion Parameter Models with Simple and Efficient Sparsity*.
- Jiang, A. Q., et al. (2024). *Mixtral of Experts*.
- DeepSeek-AI (2024). *DeepSeek-V3 Technical Report*.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — المعماريات الحديثة / الفصل 15: خليط الخبراء: Mixture of Experts.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
